# 街景字符识别 - FPN Multi-Head 模型

基于 PyTorch 的街景字符识别项目，采用 FPN + Multi-Head Attention 架构。

**支持三种模型**：`fpn_multihead`（默认）、`transformer`、`ctc`

**支持两种GPU环境**：
- NVIDIA A100 (8核 / 32GB内存 / 24GB显存)
- AMD ROCm (8核 / 200GB内存 / 192GB显存)

运行时自动检测GPU环境并选择对应的参数配置（GPUProfile策略模式）。

**torch.compile 支持**：AMD ROCm 环境下自动启用 `torch.compile`，首次运行需编译 kernel（约5-15分钟），后续运行使用缓存。

**日志路径**：`{项目目录}/logs/train_YYYYMMDD_HHMMSS.log`

In [ ]:
# 环境配置：设置随机种子、检测GPU、打印版本信息
import torch as t
import sys
import os

from utils.seed import set_seed
from config import config, BASE_DIR, data_dir, IS_MODELSCOPE, GPU_PLATFORM, TOTAL_VRAM_GB, IS_NVIDIA, IS_AMD
from utils.platform import get_precision_config, is_nvidia_cuda

# 平台特定精度配置
precision_config = get_precision_config()
if is_nvidia_cuda():
    try:
        t.set_float32_matmul_precision('high')
        print('TF32 matmul precision: enabled')
    except Exception:
        print('TF32 matmul precision: not supported on this GPU')
    try:
        t.backends.cudnn.allow_tf32 = True
        print('CuDNN TF32: enabled')
    except Exception:
        print('CuDNN TF32: not supported')
else:
    print('TF32/CuDNN: platform-specific settings skipped')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

set_seed(42)

print('=' * 60)
print('环境信息')
print('=' * 60)
print(f'Python: {sys.version}')
print(f'PyTorch: {t.__version__}')
print(f'GPU Platform: {GPU_PLATFORM.upper()}')
print(f'CUDA available: {t.cuda.is_available()}')
if t.cuda.is_available():
    print(f'CUDA version: {t.version.cuda}')
    print(f'GPU: {t.cuda.get_device_name(0)}')
    props = t.cuda.get_device_properties(0)
    vram = getattr(props, 'total_mem', getattr(props, 'total_memory', 0)) / (1024**3)
    print(f'GPU VRAM: {vram:.1f} GB')
print(f'BASE_DIR: {BASE_DIR}')
print(f'ModelScope环境: {IS_MODELSCOPE}')
print(f'Model type: {config.model_type}')
print(f'Batch size: {config.batch_size}')
print(f'Eval batch size: {config.eval_batch_size}')
print(f'Num workers: {config.num_workers}')
print(f'Gradient accumulation: {config.grad_accum_steps}')
print(f'Equivalent batch: {config.batch_size * config.grad_accum_steps}')
print(f'Input size: {config.input_height}x{config.input_width}')
print(f'Use torch.compile: {config.use_torch_compile}')
if config.use_torch_compile:
    print(f'Compile mode: {config.compile_mode}')
    print(f'Compile dynamic: {config.compile_dynamic}')
    print(f'Compile fullgraph: {config.compile_fullgraph}')
print(f'Gradient checkpoint: {config.use_gradient_checkpoint}')
print(f'Optimizer: {config.optimizer_type}')
print(f'Scheduler: {config.scheduler_type}')
print(f'Learning rate: {config.lr}')
print(f'Warmup epochs: {config.warmup_epochs}')
print(f'cls_loss_weight: {config.cls_loss_weight}')
print(f'bbox_loss_weight: {config.bbox_loss_weight}')
print(f'aux_loss_weight: {config.aux_loss_weight}')
print(f'ordering_loss_weight: {config.ordering_loss_weight}')
print(f'attn_supervision_weight: {config.attn_supervision_weight}')
print(f'attn_diversity_weight: {config.attn_diversity_weight}')
print(f'num_attn_channels: {config.num_attn_channels}')
print(f'soft_attn_temperature: {config.soft_attn_temperature}')
print(f'CutMix prob: {config.cutmix_prob}')
print(f'EMA decay: {config.ema_decay}')
print(f'Dropout: {config.dropout}')
print(f'Early stopping patience: {config.early_stopping_patience}')
print('=' * 60)

In [ ]:
# 下载数据集（仅在数据不存在时执行）
from data.download import download_dataset
download_dataset()

## 数据探索

In [ ]:
# 统计训练集、验证集、测试集的图片数量
from glob import glob

train_list = glob(data_dir['train_data'] + '*.png')
val_list = glob(data_dir['val_data'] + '*.png')
test_list = glob(data_dir['test_data'] + '*.png')

print(f'训练集图片数: {len(train_list)}')
print(f'验证集图片数: {len(val_list)}')
print(f'测试集图片数: {len(test_list)}')

In [ ]:
# 分析训练集中数字位数的分布
import json

with open(data_dir['train_label'], 'r') as f:
    marks = json.load(f)

dicts = {}
for img, mark in marks.items():
    n = len(mark['label'])
    dicts[n] = dicts.get(n, 0) + 1
for k, v in sorted(dicts.items()):
    print(f'{k}位数字的图片数目: {v}')

## 模型定义

创建模型并统计参数量

In [ ]:
# 创建模型并统计参数量
from models import create_model

model = create_model('fpn_multihead')
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'模型类型: fpn_multihead')
print(f'总参数量: {total_params:,}')
print(f'可训练参数量: {trainable_params:,}')
print(f'模型大小: {total_params * 4 / (1024**2):.1f} MB (FP32)')

## 训练 FPN Multi-Head 模型

训练过程中日志会自动记录到 `logs/` 目录下，包含：
- 每个batch的loss、学习率、精度、GPU/CPU内存使用
- 每个epoch的训练/验证精度（joint acc + char acc）、耗时、early stopping状态
- v5新增：每个epoch开始前的LR、roi_gt_prob、generator_seed诊断日志

**torch.compile warmup 说明**：
- AMD ROCm 环境下自动启用 `torch.compile(mode='default', dynamic=True)`
- 首次运行需编译 CUDA kernel，warmup 阶段约需 **5-15 分钟**（取决于缓存状态）
- 后续运行使用 inductor_cache 缓存，warmup 时间大幅缩短
- 编译失败时自动 fallback 到 eager 模式，不影响训练

In [ ]:
# 训练FPN Multi-Head模型
# 日志自动保存到 {项目目录}/logs/ 目录
#
# resume_weights_only=False (v4默认): 完整恢复optimizer/scheduler/scaler状态继续训练
#   - 适用于中断后原参数继续训练
#   - 加载train_model权重（非EMA），恢复完整训练状态
# resume_weights_only=True: 只加载模型权重，用新optimizer从头训练
#   - 适用于切换优化器/调度器/超参数后重新训练
#   - 加载EMA权重到训练模型，optimizer/scheduler/scaler全部重新初始化
from trainer.multihead import MultiHeadTrainer
from utils.misc import find_latest_checkpoint

latest = find_latest_checkpoint(config.checkpoints)
if latest:
    config.pretrained = latest
    config.resume_weights_only = False
    print(f'从最新 checkpoint 续训: {latest}')
else:
    config.pretrained = None
    config.start_epoch = 0
    print('无 checkpoint，从头训练')

trainer = MultiHeadTrainer(model_type='fpn_multihead')
trainer.train()

## 训练诊断（v7增强）

验证数据加载是否每epoch不同，LR调度是否正确，模型参数是否在更新，zero_grad是否正确执行，以及cls_loss的valid_mask过滤是否真正生效。

In [ ]:
# 诊断：验证DataLoader每epoch数据顺序不同
import torch as t
from utils.seed import make_epoch_generator

print('=== DataLoader Shuffle 诊断 ===')
print(f'训练集大小: {len(trainer.train_set)}')
print(f'Batch size: {config.batch_size}')

first_indices = []
for epoch in range(3):
    gen = make_epoch_generator(42, epoch=epoch)
    sampler = t.utils.data.RandomSampler(trainer.train_set, generator=gen)
    indices = list(sampler)[:config.batch_size]
    first_indices.append(indices)
    print(f'Epoch {epoch}: 前{config.batch_size}个样本索引 = {indices[:5]}...')

if first_indices[0] == first_indices[1]:
    print('⚠️ 警告: epoch 0 和 epoch 1 的数据顺序相同！')
else:
    print('✅ 每epoch数据顺序不同，shuffle正常')

In [ ]:
# 诊断：验证LR调度是否正确（v5新增）
print('=== LR Scheduler 诊断 ===')
print(f'Scheduler type: {config.scheduler_type}')
print(f'Base LR: {config.lr}')
print(f'Backbone LR factor: {config.backbone_lr_factor}')
print(f'Warmup epochs: {config.warmup_epochs}')
print(f'Scheduler last_epoch: {trainer.lr_scheduler.last_epoch}')
print(f'Current backbone LR: {trainer.optimizer.param_groups[0]["lr"]:.8f}')
print(f'Current head LR: {trainer.optimizer.param_groups[1]["lr"]:.8f}')
print()
print('LR schedule (first 10 epochs):')
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, CosineAnnealingWarmRestarts, SequentialLR
test_opt = t.optim.SGD([{'params': [t.randn(1)], 'lr': config.lr * config.backbone_lr_factor},
                         {'params': [t.randn(1)], 'lr': config.lr}], momentum=0.9)
if config.scheduler_type == 'warm_restarts':
    warmup = LinearLR(test_opt, start_factor=0.1, total_iters=config.warmup_epochs)
    restart = CosineAnnealingWarmRestarts(test_opt, T_0=5, T_mult=2, eta_min=1e-6)
    test_sched = SequentialLR(test_opt, schedulers=[warmup, restart], milestones=[config.warmup_epochs])
else:
    warmup = LinearLR(test_opt, start_factor=0.1, total_iters=config.warmup_epochs)
    cosine = CosineAnnealingLR(test_opt, T_max=config.epoches - config.warmup_epochs, eta_min=1e-6)
    test_sched = SequentialLR(test_opt, schedulers=[warmup, cosine], milestones=[config.warmup_epochs])
for e in range(min(10, config.epoches)):
    if e > 0:
        test_sched.step()
    print(f'  Epoch {e+1}: backbone_lr={test_opt.param_groups[0]["lr"]:.8f}, head_lr={test_opt.param_groups[1]["lr"]:.8f}')

In [ ]:
# 诊断：验证模型参数在训练中是否更新
raw = trainer._get_raw_model()
first_param = next(iter(raw.parameters()))
print(f'模型首个参数 shape: {first_param.shape}')
print(f'首个参数前5个值: {first_param.data.flatten()[:5].tolist()}')
print(f'参数范数: {first_param.data.norm().item():.4f}')
print()
total_norm = sum(p.data.norm().item() ** 2 for p in raw.parameters()) ** 0.5
print(f'模型总参数范数: {total_norm:.2f}')
print()
if trainer.train_log:
    print('最近5条训练日志:')
    for entry in trainer.train_log[-5:]:
        print(f'  Epoch {entry["epoch"]}: train_joint={entry["train_joint_acc"]:.2f}, '
              f'val_joint={entry["val_joint_acc"]:.2f}, lr={entry["lr"]:.6f}')

In [ ]:
# 诊断（v6新增）：验证zero_grad在grad_accum_steps=1时正确执行
print('=== zero_grad 诊断 (v6关键Bug验证) ===')
print(f'grad_accum_steps = {config.grad_accum_steps}')

# 验证条件逻辑
for steps in [1, 2, 4]:
    zero_grad_batches = [i for i in range(10) if i % steps == 0]
    step_batches = [i for i in range(10) if (i + 1) % steps == 0 or (i + 1) == 10]
    print(f'  grad_accum_steps={steps}: zero_grad at batch {zero_grad_batches}, step at batch {step_batches}')

# 验证旧条件（Bug）vs 新条件（修复）
print()
print('旧条件 (i+1)%grad_accum_steps==1 在 grad_accum_steps=1 时:')
old_trigger = [(i, (i+1)%1==1) for i in range(5)]
print(f'  {old_trigger} → zero_grad永远不会触发！')
print()
print('新条件 i%grad_accum_steps==0 在 grad_accum_steps=1 时:')
new_trigger = [(i, i%1==0) for i in range(5)]
print(f'  {new_trigger} → zero_grad每个batch都正确触发 ✅')

In [ ]:
# 诊断（v7新增）：验证cls_loss的valid_mask过滤是否真正生效
import torch as t
import torch.nn.functional as F
from losses.classification import LabelSmoothEntropy

print('=== cls_loss valid_mask 诊断 (v7关键Bug验证) ===')

B, C = 8, 11
pred = t.randn(B, C)
label = t.tensor([3, 7, 0, 10, 10, 10, 5, 10])
true_lengths = t.tensor([4, 3, 1, 2, 1, 1, 6, 1])

criteria_mean = LabelSmoothEntropy(smooth=0.1, size_average='mean')
criteria_none = LabelSmoothEntropy(smooth=0.1, size_average='none')

for h in range(6):
    valid_mask = (true_lengths > h).float()
    empty_mask = 1.0 - valid_mask
    n_valid = int(valid_mask.sum().item())
    n_empty = int(empty_mask.sum().item())
    
    head_loss_scalar = criteria_mean(pred, label)  # 旧: 标量
    head_loss_per_sample = criteria_none(pred, label)  # 新: 逐样本
    
    old_result = (head_loss_scalar * valid_mask).sum() / valid_mask.sum() if n_valid > 0 else 0
    new_result = (head_loss_per_sample * valid_mask).sum() / valid_mask.sum() if n_valid > 0 else 0
    
    print(f'Head {h}: valid={n_valid}, empty={n_empty}')
    print(f'  旧(标量*mask no-op): {old_result.item():.6f}')
    print(f'  新(逐样本*mask过滤): {new_result.item():.6f}')
    if n_valid > 0 and n_empty > 0:
        ratio = new_result.item() / old_result.item() if old_result.item() != 0 else float('inf')
        print(f'  新/旧比值: {ratio:.2f}x (空位越多比值越大)')
    print()

## 评估

In [ ]:
# 在验证集上评估模型精度（返回joint acc）
val_acc = trainer._eval()
print(f'最佳验证精度: {trainer.best_acc * 100:.2f}%')

In [ ]:
# 逐Head评估每个数字位的分类精度
trainer.eval_detailed()

## torch.compile 审计（可选）

运行系统性编译审计，覆盖编译模式、输入数据类型/尺寸/批次、warmup策略对比、正确性验证等。
完整审计约需 1-2 小时，可按需选择单项审计。

In [ ]:
# 运行 torch.compile 系统性审计（可选，取消注释执行）
# !python scripts/compile_audit.py --audit warmup  # 仅测试warmup策略
# !python scripts/compile_audit.py --audit modes   # 仅测试编译模式
# !python scripts/compile_audit.py --audit all     # 完整审计

## 推理与提交

In [ ]:
# 推理（use_compile=True 时启用 torch.compile 加速推理）
from inference.predict import predicts, ctc_predict, ensemble_predict

# model_path = 'checkpoints/best-resnet101-acc-xx.xx.pth'
# results = predicts(model_path, 'submit.csv', use_tta=True, use_compile=True)

In [ ]:
# TTA 评估
# tta_acc = trainer.eval_tta()
# print(f'TTA Accuracy: {tta_acc * 100:.2f}%')

## 训练日志

In [ ]:
# 查看训练日志
for entry in trainer.train_log[-5:]:
    print(entry)

In [ ]:
# 保存模型
# trainer.save_model('checkpoints/manual_save.pth', save_opt=True)